# kNN

In [ ]:
%run preprocessing.ipynb

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nbimporter

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score
import numpy as np

# --- Pipelines ---
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_cols),
    ('cat', categorical_pipeline, categorical_cols),
])

# --- Labels ---
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_raw)
y_val   = label_encoder.transform(y_val_raw)
y_test  = label_encoder.transform(y_test_raw)

# --- Encode ---
X_train_enc = preprocessor.fit_transform(X_train)
X_val_enc   = preprocessor.transform(X_val)
X_test_enc  = preprocessor.transform(X_test)

# --- PCA ---
pca = PCA(n_components=0.95)
X_train_prepared = pca.fit_transform(X_train_enc)
X_val_prepared   = pca.transform(X_val_enc)
X_test_prepared  = pca.transform(X_test_enc)

# --- KNN loop ---
k = [1, 3, 5, 10, 20, 50, 100, 150]
training_errors = []
testing_errors = []

figure, axes = plt.subplots(3, figsize=(20, 20))

k_vals = [1, 3, 5, 10, 20, 50, 100, 200]
training_errors = []
testing_errors = []

for i in k_vals:
    knn = KNeighborsClassifier(n_neighbors=i)
    knn.fit(X_train_prepared, y_train)

    # --- TRAIN ---
    y_pred = knn.predict(X_train_prepared)
    acc = accuracy_score(y_train, y_pred)
    training_errors.append({
        "k": i,
        "Accuracy": acc,
        "Error": 1-acc
        "Recall": recall_score(y_train, y_pred, average='weighted'),
        "F1 Score": f1_score(y_train, y_pred, average='weighted')
    })

    # --- VALIDATION ---
    y_pred = knn.predict(X_val_prepared)
    acc = accuracy_score(y_val, y_pred)
    testing_errors.append({
        "k": i,
        "Accuracy": acc,
        "Error": 1-acc,
        "Recall": recall_score(y_val, y_pred, average='weighted'),
        "F1 Score": f1_score(y_val, y_pred, average='weighted')
    })

testing_df = pd.DataFrame(testing_errors)
print(testing_df)
training_df = pd.DataFrame(training_errors)
print(training_df)

figure, axes = plt.subplots(3, figsize=(20, 20))

metrics = ['Accuracy', 'Error', 'Recall', 'F1 Score']

for i in range(4):
    value = metrics[i]

    axes[i].set_xlabel("k")
    axes[i].set_ylabel(value)

    axes[i].semilogx(testing_df["k"], testing_df[value], label="validation")
    axes[i].semilogx(training_df["k"], training_df[value], label="training")

    axes[i].set_title(f'{value} vs k')
    axes[i].legend()